# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue

---

and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
print('Sum of total revenue is:', df['revenue'].sum())

Sum of total revenue is: 8520.0


total revenue is $8520.0

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = df.groupby('category')['revenue'].sum().to_frame()
by_category['share_pct'] = (by_category['revenue'] / by_category['revenue'].sum() * 100).round(1)
by_category.sort_values('revenue', ascending=False)

,revenue,share_pct
category,,
Food,4293.0,50.4
Merch,1771.5,20.8
Drink,1554.0,18.2
RainGear,901.5,10.6


Food has by far the highest revenue share percentage, showing that it is the most "important" category for vendors to stock and sell. Rain gear is a very condition-based item.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = df.groupby('vendor_id')['revenue'].agg(avg_revenue='mean', order_count='count')
by_vendor.sort_values('avg_revenue', ascending=False)

,avg_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


V-01 has the highest average order revenue while having the second lowest orders. The relatively low number of orders may help them have a higher average.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
print(f'{by_category.loc["Merch", "revenue"] / df["revenue"].sum():.1%}')
print(by_category.loc['Merch','share_pct'])

20.8%
20.8


20.8% of sales come from merch. The same information is seen in the table from question 2.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

#TODO

# prove row count and revenue total are unchanged. same assertion in Q7
assert len(joined) == len(df)
assert np.isclose(joined['revenue'].sum(), df['revenue'].sum())

# unmatched vendor
unmatched = joined[joined['vendor_name'].isna()]['vendor_id'].unique()
print('Unmatched vendor_id(s):', unmatched)

joined['vendor_name'] = joined['vendor_name'].fillna(joined['vendor_id'])
joined.head()

Unmatched vendor_id(s): ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,V-18
2,V-18,Drink,3,4.5,13.5,V-18
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,V-18


**The unmatched vendor, and what I did about it:** _..._ I matched the vendor with itself instead of filling the unknown vendor as NaN. Because there is a vendor, and that vendor is not changing, it felt unnecessary to fill it as NaN, as to me, that implies that the vendor "doesn't exist", when it very much does.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
piv = pd.pivot_table(
    joined, values='revenue', index='vendor_name', columns='category',
    aggfunc='sum', margins=True, fill_value=0
)
piv

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
V-18,582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


Pivot table allows for a very easy understanding of revenue. It should probably be titled to indicate that the numbers represent revenue by category. It would also be interesting to see the pivot table as only percentages...

In [8]:
pivot_pct = (piv / piv.loc['All', 'All'] * 100).round(1)
pivot_pct

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,5.9,12.4,4.7,2.1,25.0
Hoos Burgers,2.0,15.7,4.4,2.8,24.9
Rotunda Tacos,3.5,10.4,5.7,2.9,22.5
V-18,6.8,12.0,6.0,2.8,27.6
All,18.2,50.4,20.8,10.6,100.0


pivot table as percentages.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [9]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A: Looking at the table from question 2, it is clear that food is the biggest driver of revenue out of all streams (50.4% from all vendors). It is clear that vendors should prioritize their food sales and stock in order to maximize revenue. It should also be noted that items such as rain gear are very situational and almost wholly dependent on the weather conditions. Assuming that this data comes from a fair weather game, if fair weather is expected, vendors may be able to reduce stock of rain gear, as it only accounts for 10.6% of all sales. This would potentially allow them to focus on stocking other items. My final suggestion is likely hard to implement with a week's lead, but I would tell Rotunda Taco's that they either need to improve their food, or fix their branding. They made a total of $882.0, which is the lowest of all vendors from a total share of food sales. This number suggests that they are either being beat in the marketing department, or their food is bad compared to their competitors.

In [10]:
food_total = joined.loc[joined['category'] == 'Food', 'revenue'].sum()

food_by_vendor_sum = (
    joined[joined['category'] == 'Food']
    .groupby('vendor_name')['revenue']
    .sum()
    .to_frame(name='food_revenue')
)
food_by_vendor_sum['share_pct'] = (
    food_by_vendor_sum['food_revenue'] / food_total * 100
).round(1)

food_by_vendor_sum.sort_values('food_revenue', ascending=False)

,food_revenue,share_pct
vendor_name,,
Hoos Burgers,1338.0,31.2
Cav Merch North,1054.5,24.6
V-18,1018.5,23.7
Rotunda Tacos,882.0,20.5


B: From a utility perspective (how useful the data is as a report), having V-18 as the vendor name is pretty unhelpful, but not untrustworthy. As I mentioned in the question, the numbers are valid, but not knowing who V-18 is would be pretty annoying if you are trying to track them down. In terms of being untrustworthy, I think the average revenue by highest average order revenue is pretty misleading, as it somewhat ignores what the vendors are actually selling, and it doesn't provide much utility in understanding who is making the most money. For example, if a vendor only sold 5 limited edition jersey for $100, their average revenue would be the highest of the vendors, but they would have the least revenue. I don't think the order count is the problem (in terms of small sample size), rather the data isn't *that* useful.  

In [11]:
order_counts = pd.pivot_table(
    joined, values='revenue', index='vendor_name', columns='category',
    aggfunc='count', margins=True, fill_value=0
)
order_counts

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,29,51,16,9,105
Hoos Burgers,12,55,17,10,94
Rotunda Tacos,17,37,24,15,93
V-18,31,43,22,12,108
All,89,186,79,46,400


In [12]:
avg_price_by_category = (
    df.groupby('category')['price']
    .mean()
    .round(2)
    .sort_values(ascending=False)
)
avg_price_by_category

,price
category,
Food,11.76
Merch,11.24
RainGear,11.15
Drink,9.15
